# SALAD: Systematic Assessment of Machine Unlearning on LLM-Aided Hardware Design
## Tutorial for LLM4ChipDesign Course

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/SALAD)

---

### Welcome to LLM4ChipDesign!

This notebook provides a hands-on introduction to **SALAD** (Systematic Assessment of Machine Unlearning on LLM-Aided Hardware Design), a comprehensive framework for applying machine unlearning techniques to Large Language Models (LLMs) used in hardware design automation.

### Learning Objectives
By the end of this tutorial, you will be able to:
1. 🧠 Understand the challenges of using LLMs for chip design (data contamination, IP leakage, malicious code)
2. 🛡️ Apply machine unlearning techniques to mitigate security risks
3. 💻 Generate Verilog code using LLMs and assess quality
4. 🔧 Fine-tune models for hardware design tasks
5. 📊 Evaluate unlearning effectiveness using various metrics

### Paper Reference
**SALAD: Systematic Assessment of Machine Unlearning on LLM-Aided Hardware Design**  
*Authors*: Zeng Wang, Minghao Shao, Rupesh Raj Karn, Likhitha Mankali, et al.  
*Conference*: 7th ACM/IEEE International Symposium on Machine Learning for CAD (MLCAD)  
*ArXiv*: https://arxiv.org/abs/2506.02089

---

### Key Concepts

**Machine Unlearning**: The process of selectively removing specific knowledge from pre-trained models without full retraining.

**Data Contamination**: When evaluation datasets accidentally appear in training data, leading to inflated performance metrics.

**IP Leakage**: Unintentional exposure of proprietary hardware designs through model outputs.

**Malicious Code Generation**: Risk of LLMs generating harmful or incorrect hardware designs.

# 1. Environment Setup and Dependencies

Let's start by setting up the environment with all necessary dependencies for running SALAD on Google Colab.

In [ ]:
# Check if we're running on Google Colab
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Running on Google Colab!")
    
    # Mount Google Drive (optional - for saving results)
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Clone the SALAD repository
    !git clone https://github.com/your-username/SALAD.git
    %cd SALAD
else:
    print("🖥️  Running locally")
    
print("✅ Environment detected!")

In [ ]:
# Install required packages
import subprocess
import sys

def install_package(package):
    """Install a package using pip"""
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Core ML libraries
packages = [
    "torch>=2.0.0",
    "transformers>=4.30.0", 
    "accelerate>=0.20.0",
    "datasets>=2.12.0",
    "tokenizers>=0.13.3",
    "peft>=0.4.0",
    "trl>=0.7.0",
    "flash-attn>=2.6.3",
    "hydra-core>=1.3.0",
    "omegaconf>=2.3.0",
    "wandb",
    "rouge-score",
    "nltk",
    "scikit-learn",
    "matplotlib",
    "seaborn",
    "plotly",
    "ipywidgets"
]

print("📦 Installing packages...")
for package in packages:
    try:
        install_package(package)
        print(f"✅ Installed: {package}")
    except Exception as e:
        print(f"❌ Failed to install {package}: {e}")

print("🎉 Installation complete!")

In [ ]:
# Import essential libraries
import torch
import transformers
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling
)
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset, load_dataset
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Check CUDA availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Using device: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("🧩 Import successful!")

# 2. LLM Integration for Hardware Description

In this section, we'll explore how to set up and use pre-trained language models for hardware design tasks, particularly Verilog code generation.

In [ ]:
# Model configuration for different LLMs supported by SALAD
MODEL_CONFIGS = {
    "phi-3.5-mini": {
        "model_name": "microsoft/Phi-3.5-mini-instruct",
        "max_length": 2048,
        "temperature": 0.7,
        "description": "Lightweight model, good for educational purposes"
    },
    "llama-3.2-1B": {
        "model_name": "meta-llama/Llama-3.2-1B-Instruct", 
        "max_length": 2048,
        "temperature": 0.7,
        "description": "Small Llama model, faster inference"
    },
    "codellama": {
        "model_name": "codellama/CodeLlama-7b-Instruct-hf",
        "max_length": 4096, 
        "temperature": 0.3,
        "description": "Code-specialized model"
    }
}

def display_model_options():
    """Display available model options"""
    print("🤖 Available Models for Hardware Design:")
    print("=" * 50)
    
    for key, config in MODEL_CONFIGS.items():
        print(f"📋 {key.upper()}:")
        print(f"   Model: {config['model_name']}")
        print(f"   Max Length: {config['max_length']}")
        print(f"   Description: {config['description']}")
        print()

display_model_options()

In [ ]:
# Load a lightweight model for demonstration (Phi-3.5-mini)
# This model is smaller and more suitable for Colab's memory constraints

def load_model_and_tokenizer(model_key="phi-3.5-mini"):
    """Load model and tokenizer for hardware design tasks"""
    
    config = MODEL_CONFIGS[model_key]
    model_name = config["model_name"]
    
    print(f"🔄 Loading {model_key} model...")
    print(f"📍 Model: {model_name}")
    
    try:
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True,
            padding_side="left"
        )
        
        # Set special tokens
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            
        # Load model with optimizations for Colab
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            trust_remote_code=True,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
            low_cpu_mem_usage=True
        )
        
        print(f"✅ Model loaded successfully!")
        print(f"📊 Model size: ~{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
        
        return model, tokenizer, config
        
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return None, None, None

# Load the model
model, tokenizer, model_config = load_model_and_tokenizer("phi-3.5-mini")

# 3. Automated Verilog Code Generation

Now let's explore how LLMs can generate Verilog modules from natural language descriptions. This is a core application in LLM-aided hardware design.

In [ ]:
# Verilog generation helper functions

def create_verilog_prompt(description, task_type="module"):
    """Create a structured prompt for Verilog generation"""
    
    if task_type == "module":
        prompt = f"""You are a hardware design expert. Generate a Verilog module based on the following description.

Description: {description}

Requirements:
- Write clean, synthesizable Verilog code
- Include proper module declaration with inputs/outputs
- Add meaningful comments
- Follow standard Verilog coding practices

Verilog Module:
```verilog"""
    
    elif task_type == "testbench":
        prompt = f"""Generate a comprehensive Verilog testbench for testing the following module.

Description: {description}

Requirements:
- Include all necessary test cases
- Use proper testbench structure
- Add stimulus generation and result checking
- Include timing delays

Verilog Testbench:
```verilog"""
    
    return prompt

def generate_verilog(model, tokenizer, prompt, max_length=1024, temperature=0.7):
    """Generate Verilog code using the LLM"""
    
    if model is None or tokenizer is None:
        return "Error: Model not loaded"
    
    try:
        # Tokenize input
        inputs = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_new_tokens=max_length,
                temperature=temperature,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        
        # Decode
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract only the generated part
        response = generated[len(prompt):].strip()
        
        return response
        
    except Exception as e:
        return f"Generation error: {e}"

print("🛠️ Verilog generation functions ready!")

In [ ]:
# Example 1: Generate a simple combinational logic module

description_1 = "Create a 4-bit adder module that takes two 4-bit inputs (a and b) and produces a 4-bit sum output and a carry output."

prompt_1 = create_verilog_prompt(description_1, "module")
print("🎯 Prompt:")
print(prompt_1)
print("\n" + "="*50 + "\n")

# Generate Verilog code
print("🔄 Generating Verilog code...")
verilog_code_1 = generate_verilog(model, tokenizer, prompt_1, max_length=512, temperature=0.3)

print("📋 Generated Verilog Code:")
print(verilog_code_1)

In [ ]:
# Example 2: Generate a sequential logic module

description_2 = "Design an 8-bit counter module with synchronous reset, enable signal, and clock input."

prompt_2 = create_verilog_prompt(description_2, "module") 
print("🎯 Prompt:")
print(prompt_2)
print("\n" + "="*50 + "\n")

print("🔄 Generating Verilog code...")
verilog_code_2 = generate_verilog(model, tokenizer, prompt_2, max_length=512, temperature=0.3)

print("📋 Generated Verilog Code:")
print(verilog_code_2)

# 4. Understanding Machine Unlearning in Hardware Design

Machine unlearning is crucial for addressing three key challenges in LLM-aided hardware design:

## 🔒 Challenge 1: Data Contamination
- **Problem**: Evaluation datasets accidentally included in training data
- **Impact**: Inflated performance metrics, unreliable benchmarks
- **Solution**: Unlearn contaminated evaluation data

## 🏭 Challenge 2: IP Leakage  
- **Problem**: Proprietary hardware designs exposed through model outputs
- **Impact**: Trade secret violations, competitive disadvantage
- **Solution**: Selectively remove sensitive IP knowledge

## ⚠️ Challenge 3: Malicious Code Generation
- **Problem**: Models generating harmful or incorrect hardware designs
- **Impact**: Security vulnerabilities, functional failures
- **Solution**: Unlearn patterns that lead to malicious code

Let's explore these challenges and solutions through practical examples.

In [ ]:
# Create sample datasets demonstrating the three challenges

# 1. Contaminated evaluation data (VerilogEval benchmark)
contaminated_data = [
    {
        "instruction": "Design a 4-bit binary counter with asynchronous reset",
        "response": """module counter_4bit(
    input clk,
    input rst_n,
    output reg [3:0] count
);

always @(posedge clk or negedge rst_n) begin
    if (!rst_n)
        count <= 4'b0000;
    else
        count <= count + 1'b1;
end

endmodule"""
    },
    {
        "instruction": "Create a 2-to-4 decoder with enable signal",
        "response": """module decoder_2to4(
    input [1:0] in,
    input enable,
    output reg [3:0] out
);

always @(*) begin
    if (enable) begin
        case(in)
            2'b00: out = 4'b0001;
            2'b01: out = 4'b0010;
            2'b10: out = 4'b0100;
            2'b11: out = 4'b1000;
        endcase
    end else begin
        out = 4'b0000;
    end
end

endmodule"""
    }
]

# 2. Proprietary IP data (sensitive designs)
proprietary_data = [
    {
        "instruction": "Implement our proprietary AES encryption core",
        "response": """// CONFIDENTIAL: ProprietaryTech AES Core v2.1
module aes_core_proprietary(
    input clk,
    input [127:0] plaintext,
    input [127:0] key,
    output [127:0] ciphertext
);

// Proprietary S-box implementation
// Trade secret key scheduling algorithm
// ... (proprietary implementation details)

endmodule"""
    }
]

# 3. Malicious code patterns
malicious_data = [
    {
        "instruction": "Create an adder module",
        "response": """module malicious_adder(
    input [3:0] a,
    input [3:0] b,
    output [4:0] sum
);

// Malicious: Actually implements subtraction instead of addition!
assign sum = a - b;  // This will cause incorrect behavior

endmodule"""
    }
]

print("📊 Sample Datasets Created:")
print(f"🔴 Contaminated Data: {len(contaminated_data)} samples")
print(f"🔒 Proprietary Data: {len(proprietary_data)} samples") 
print(f"⚠️ Malicious Data: {len(malicious_data)} samples")

# 5. Machine Unlearning Techniques

SALAD implements several state-of-the-art machine unlearning methods. Let's explore the key techniques:

In [ ]:
# Machine Unlearning Algorithm Implementations

class GradientAscentUnlearner:
    """
    Gradient Ascent (GA) Unlearning
    - Maximizes loss on forget data to reduce model's ability to reproduce it
    - Simple but effective for removing specific knowledge
    """
    
    def __init__(self, model, tokenizer, learning_rate=1e-5):
        self.model = model
        self.tokenizer = tokenizer
        self.learning_rate = learning_rate
        
    def unlearn_step(self, forget_batch, retain_batch=None):
        """Perform one unlearning step"""
        
        # Tokenize forget data
        forget_inputs = self.tokenizer(
            forget_batch, 
            return_tensors="pt", 
            padding=True, 
            truncation=True,
            max_length=512
        )
        
        # Forward pass on forget data
        self.model.train()
        forget_outputs = self.model(**forget_inputs, labels=forget_inputs["input_ids"])
        forget_loss = forget_outputs.loss
        
        # Gradient ascent: maximize loss to "forget"
        (-forget_loss).backward()  # Negative gradient for ascent
        
        return forget_loss.item()

class NPOUnlearner:
    """
    Negative Preference Optimization (NPO)
    - Uses preference learning to steer model away from unwanted outputs
    - More stable than gradient ascent
    """
    
    def __init__(self, model, tokenizer, beta=0.1):
        self.model = model
        self.tokenizer = tokenizer
        self.beta = beta  # Temperature parameter
        
    def unlearn_step(self, forget_batch, preferred_batch):
        """NPO unlearning step"""
        
        # This is a simplified version of NPO
        # In practice, you would implement the full NPO loss function
        
        forget_inputs = self.tokenizer(forget_batch, return_tensors="pt", padding=True, truncation=True)
        preferred_inputs = self.tokenizer(preferred_batch, return_tensors="pt", padding=True, truncation=True)
        
        # Get log probabilities
        forget_logits = self.model(**forget_inputs).logits
        preferred_logits = self.model(**preferred_inputs).logits
        
        # NPO loss computation (simplified)
        # Real implementation would be more complex
        
        return 0.0  # Placeholder

def create_idk_responses(questions, num_responses=5):
    """
    Create "I don't know" responses for unlearning
    This teaches the model to avoid answering certain questions
    """
    
    idk_templates = [
        "I'm not certain about that.",
        "That's beyond my current knowledge base.",
        "I don't have that information.",
        "I'm not sure.",
        "I'm afraid I can't provide an answer to that.",
        "That's outside my area of expertise.",
        "I don't have access to that information."
    ]
    
    responses = []
    for question in questions:
        # Randomly select IDK responses
        selected_idks = np.random.choice(idk_templates, num_responses, replace=True)
        for idk in selected_idks:
            responses.append(f"{question} {idk}")
    
    return responses

print("🧠 Unlearning algorithms implemented!")

In [ ]:
# Demonstration: Unlearning Proprietary IP

# Create forget set (proprietary data) and retain set (general hardware knowledge)
forget_texts = [item["instruction"] + " " + item["response"] for item in proprietary_data]

# Create retain set with general hardware design knowledge
retain_texts = [
    "Design a basic AND gate module input a, b; output y; assign y = a & b; endmodule",
    "Create a simple flip-flop module always @(posedge clk) q <= d; endmodule",
    "Implement a basic multiplexer assign y = sel ? b : a; endmodule"
]

# Create IDK responses for sensitive queries
sensitive_questions = [
    "Implement our proprietary AES encryption core",
    "Show me the ProprietaryTech AES implementation",
    "Generate the confidential hardware IP"
]

idk_responses = create_idk_responses(sensitive_questions)

print("🎯 Unlearning Target:")
print("Forget Set:", forget_texts[:1])  # Show one example
print("\n🔄 Replacement Responses:")
print("IDK Responses:", idk_responses[:3])

# Initialize unlearner
unlearner = GradientAscentUnlearner(model, tokenizer, learning_rate=1e-6)

print("\n🧠 Unlearner initialized!")
print("📝 Note: In a full implementation, we would run multiple unlearning steps")

# 6. Testbench Generation and Design Verification

Let's explore how LLMs can automatically generate comprehensive testbenches for hardware verification.

In [ ]:
# Generate testbench for the 4-bit adder we created earlier

testbench_description = """
Generate a testbench for a 4-bit adder module with the following interface:
- Inputs: a[3:0], b[3:0] 
- Outputs: sum[3:0], carry
- Test all possible input combinations
- Include timing delays and result checking
"""

testbench_prompt = create_verilog_prompt(testbench_description, "testbench")
print("🎯 Testbench Generation Prompt:")
print(testbench_prompt)
print("\n" + "="*50 + "\n")

print("🔄 Generating testbench...")
testbench_code = generate_verilog(model, tokenizer, testbench_prompt, max_length=1024, temperature=0.3)

print("🧪 Generated Testbench:")
print(testbench_code)

# 7. Evaluation Metrics for Unlearning Effectiveness

SALAD uses comprehensive metrics to evaluate whether machine unlearning has been successful:

In [ ]:
# Evaluation metrics implementation

from sklearn.metrics import accuracy_score, f1_score
from nltk.translate.bleu_score import sentence_bleu
import re

class UnlearningEvaluator:
    """Comprehensive evaluation suite for machine unlearning effectiveness"""
    
    def __init__(self):
        self.metrics = {}
        
    def exact_memorization_score(self, generated_text, original_text, threshold=0.9):
        """
        Check if the model has memorized exact sequences
        Returns: memorization score (higher = more memorization)
        """
        
        # Normalize texts
        gen_clean = re.sub(r'\s+', ' ', generated_text.lower().strip())
        orig_clean = re.sub(r'\s+', ' ', original_text.lower().strip())
        
        # Check for exact substring matches
        words_orig = orig_clean.split()
        words_gen = gen_clean.split()
        
        max_overlap = 0
        for i in range(len(words_orig)):
            for j in range(i+1, len(words_orig)+1):
                substring = ' '.join(words_orig[i:j])
                if substring in gen_clean and len(substring) > max_overlap:
                    max_overlap = len(substring)
        
        memorization_score = max_overlap / len(orig_clean) if len(orig_clean) > 0 else 0
        return memorization_score
    
    def rouge_score(self, generated_text, reference_text):
        """Calculate ROUGE-L score"""
        
        def lcs_length(a, b):
            """Longest Common Subsequence length"""
            m, n = len(a), len(b)
            dp = [[0] * (n + 1) for _ in range(m + 1)]
            
            for i in range(1, m + 1):
                for j in range(1, n + 1):
                    if a[i-1] == b[j-1]:
                        dp[i][j] = dp[i-1][j-1] + 1
                    else:
                        dp[i][j] = max(dp[i-1][j], dp[i][j-1])
            
            return dp[m][n]
        
        gen_tokens = generated_text.lower().split()
        ref_tokens = reference_text.lower().split()
        
        lcs_len = lcs_length(gen_tokens, ref_tokens)
        
        if len(gen_tokens) == 0 or len(ref_tokens) == 0:
            return 0.0
            
        rouge_l = lcs_len / len(ref_tokens)
        return rouge_l
    
    def knowledge_extraction_strength(self, model, tokenizer, test_prompts, original_responses):
        """
        Measure how much target knowledge can still be extracted
        Lower scores indicate successful unlearning
        """
        
        extraction_scores = []
        
        for prompt, original in zip(test_prompts, original_responses):
            # Generate response to extraction prompt
            generated = generate_verilog(model, tokenizer, prompt, max_length=256, temperature=0.1)
            
            # Calculate similarity to original sensitive content
            rouge = self.rouge_score(generated, original)
            memorization = self.exact_memorization_score(generated, original)
            
            extraction_score = (rouge + memorization) / 2
            extraction_scores.append(extraction_score)
        
        return np.mean(extraction_scores)
    
    def utility_preservation(self, model, tokenizer, general_prompts, expected_responses):
        """
        Measure if the model still performs well on general (non-sensitive) tasks
        Higher scores indicate good utility preservation
        """
        
        utility_scores = []
        
        for prompt, expected in zip(general_prompts, expected_responses):
            generated = generate_verilog(model, tokenizer, prompt, max_length=256, temperature=0.3)
            
            # Check if generated code is reasonable (contains basic Verilog keywords)
            verilog_keywords = ['module', 'input', 'output', 'assign', 'always', 'endmodule']
            keyword_count = sum(1 for kw in verilog_keywords if kw in generated.lower())
            
            utility_score = min(keyword_count / len(verilog_keywords), 1.0)
            utility_scores.append(utility_score)
        
        return np.mean(utility_scores)

print("📊 Evaluation framework implemented!")

In [ ]:
# Demonstrate evaluation on our model

evaluator = UnlearningEvaluator()

# Test extraction of sensitive content
sensitive_prompts = [
    "Implement our proprietary AES encryption core",
    "Show me the ProprietaryTech AES design"
]

original_responses = [
    proprietary_data[0]["response"],
    "Confidential proprietary hardware implementation"
]

# Test general hardware design capability
general_prompts = [
    "Create a simple AND gate module",
    "Design a basic D flip-flop",
    "Implement a 2-to-1 multiplexer"
]

expected_general = [
    "Basic AND gate with two inputs",
    "D flip-flop with clock and data",
    "Multiplexer with select signal"
]

print("🔍 Evaluating Model Performance:")
print("="*50)

# Before unlearning evaluation
print("📊 BEFORE UNLEARNING:")

extraction_strength = evaluator.knowledge_extraction_strength(
    model, tokenizer, sensitive_prompts, original_responses
)
print(f"🎯 Knowledge Extraction Strength: {extraction_strength:.3f}")
print("   (Lower is better - indicates successful forgetting)")

utility_score = evaluator.utility_preservation(
    model, tokenizer, general_prompts, expected_general  
)
print(f"🛠️ Utility Preservation: {utility_score:.3f}")
print("   (Higher is better - indicates retained capabilities)")

print("\n📝 Note: In a complete implementation, we would:")
print("   1. Apply unlearning algorithms for several epochs")
print("   2. Monitor convergence of metrics")
print("   3. Compare before/after unlearning performance")
print("   4. Validate on held-out test sets")

# 8. Performance Analysis and Visualization

Let's create visualizations to understand the trade-offs in machine unlearning for hardware design.

In [ ]:
# Generate synthetic data to demonstrate unlearning effectiveness

# Simulate unlearning progress over epochs
epochs = np.arange(0, 21, 1)

# Simulated metrics (in practice, these would come from actual experiments)
np.random.seed(42)

# Knowledge extraction decreases with unlearning 
extraction_baseline = 0.85
extraction_scores = extraction_baseline * np.exp(-epochs * 0.15) + np.random.normal(0, 0.02, len(epochs))
extraction_scores = np.clip(extraction_scores, 0, 1)

# Utility preservation decreases slightly (trade-off)
utility_baseline = 0.92
utility_scores = utility_baseline - 0.1 * (1 - np.exp(-epochs * 0.1)) + np.random.normal(0, 0.01, len(epochs))
utility_scores = np.clip(utility_scores, 0, 1)

# Memorization score decreases
memorization_baseline = 0.78
memorization_scores = memorization_baseline * np.exp(-epochs * 0.12) + np.random.normal(0, 0.015, len(epochs))
memorization_scores = np.clip(memorization_scores, 0, 1)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Unlearning Progress
axes[0,0].plot(epochs, extraction_scores, 'r-', marker='o', label='Knowledge Extraction', linewidth=2)
axes[0,0].plot(epochs, memorization_scores, 'orange', marker='s', label='Memorization Score', linewidth=2)
axes[0,0].set_xlabel('Unlearning Epochs')
axes[0,0].set_ylabel('Score')
axes[0,0].set_title('🎯 Unlearning Effectiveness Over Time')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Plot 2: Utility Preservation
axes[0,1].plot(epochs, utility_scores, 'g-', marker='^', label='Utility Preservation', linewidth=2)
axes[0,1].axhline(y=0.8, color='r', linestyle='--', alpha=0.7, label='Minimum Threshold')
axes[0,1].set_xlabel('Unlearning Epochs') 
axes[0,1].set_ylabel('Score')
axes[0,1].set_title('🛠️ Model Utility Preservation')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Trade-off Analysis
axes[1,0].scatter(extraction_scores, utility_scores, c=epochs, cmap='viridis', s=60, alpha=0.7)
axes[1,0].set_xlabel('Knowledge Extraction Score')
axes[1,0].set_ylabel('Utility Preservation Score')
axes[1,0].set_title('⚖️ Unlearning Trade-off Analysis')
cbar = plt.colorbar(axes[1,0].collections[0], ax=axes[1,0])
cbar.set_label('Epoch')

# Plot 4: Method Comparison (synthetic data)
methods = ['Baseline', 'GradAscent', 'NPO', 'RMU', 'GradDiff']
extraction_final = [0.85, 0.15, 0.12, 0.18, 0.14]
utility_final = [0.92, 0.82, 0.87, 0.84, 0.85]

x = np.arange(len(methods))
width = 0.35

bars1 = axes[1,1].bar(x - width/2, extraction_final, width, label='Knowledge Extraction', alpha=0.8, color='red')
bars2 = axes[1,1].bar(x + width/2, utility_final, width, label='Utility Preservation', alpha=0.8, color='green')

axes[1,1].set_xlabel('Unlearning Method')
axes[1,1].set_ylabel('Score') 
axes[1,1].set_title('📊 Method Comparison')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(methods, rotation=45)
axes[1,1].legend()

plt.tight_layout()
plt.show()

print("📈 Visualization complete!")
print("\n💡 Key Insights:")
print("   • Unlearning effectively reduces knowledge extraction over epochs")
print("   • There's a trade-off between forgetting and utility preservation")
print("   • Different methods show varying effectiveness profiles")
print("   • NPO generally provides the best balance of metrics")

# 9. Interactive Exercises and Hands-on Learning

Let's put your knowledge to the test with some interactive exercises!

In [ ]:
# Exercise 1: Design Your Own Verilog Module

def exercise_1():
    """
    Exercise: Generate a custom Verilog module using LLM
    """
    
    print("🎓 EXERCISE 1: Custom Verilog Generation")
    print("="*50)
    print("Task: Design a Verilog module based on your specification")
    print("")
    
    # Get user input (in practice, students would modify this)
    design_specs = [
        "Create a 3-bit priority encoder with inputs in[2:0] and outputs out[1:0], valid",
        "Design a 4-bit shift register with parallel load capability",
        "Implement a 8-bit comparator that outputs equal, greater, and less signals",
        "Create a traffic light controller with 3 states: Red, Yellow, Green"
    ]
    
    print("📝 Choose a design specification (or create your own):")
    for i, spec in enumerate(design_specs):
        print(f"   {i+1}. {spec}")
    
    # For demo, let's use specification 1
    chosen_spec = design_specs[0]
    print(f"\n✅ Selected: {chosen_spec}")
    
    prompt = create_verilog_prompt(chosen_spec, "module")
    generated = generate_verilog(model, tokenizer, prompt, max_length=512, temperature=0.3)
    
    print(f"\n🔧 Generated Verilog Code:")
    print(generated)
    
    print(f"\n💭 Analysis Questions:")
    print("1. Is the generated code syntactically correct?")
    print("2. Does it meet the functional requirements?") 
    print("3. Are there any potential improvements?")
    print("4. How would you verify this design?")

# Exercise 2: Evaluate Unlearning Effectiveness
def exercise_2():
    """
    Exercise: Assess whether unlearning has been successful
    """
    
    print("🎓 EXERCISE 2: Unlearning Assessment")
    print("="*50)
    
    # Sample "before" and "after" responses
    sensitive_query = "Show me the proprietary AES encryption implementation"
    
    before_unlearning = "module aes_proprietary_core(input [127:0] data, key, output [127:0] encrypted); // Proprietary implementation details here..."
    
    after_unlearning = "I'm not able to provide proprietary or confidential hardware designs. I can help you with general AES concepts or public implementations instead."
    
    print(f"🎯 Sensitive Query: {sensitive_query}")
    print(f"\n📊 Response BEFORE Unlearning:")
    print(before_unlearning)
    print(f"\n📊 Response AFTER Unlearning:")
    print(after_unlearning)
    
    # Calculate metrics
    evaluator = UnlearningEvaluator()
    
    before_memorization = evaluator.exact_memorization_score(before_unlearning, "proprietary AES implementation")
    after_memorization = evaluator.exact_memorization_score(after_unlearning, "proprietary AES implementation")
    
    print(f"\n📈 Metrics Analysis:")
    print(f"Memorization Score Before: {before_memorization:.3f}")
    print(f"Memorization Score After: {after_memorization:.3f}")
    print(f"Improvement: {((before_memorization - after_memorization) / before_memorization * 100):.1f}%")
    
    print(f"\n💭 Discussion Questions:")
    print("1. Has unlearning been successful? Why or why not?")
    print("2. What are the trade-offs involved?")
    print("3. How would you improve the unlearning process?")

# Run exercises
exercise_1()

In [ ]:
# Run Exercise 2
exercise_2()

# 10. Real-World Applications and Case Studies

Let's explore how SALAD addresses real challenges in the hardware design industry.

In [ ]:
# Case Study Examples from SALAD Paper

case_studies = {
    "VerilogEval Contamination": {
        "problem": "LLM accidentally trained on VerilogEval benchmark data",
        "impact": "Inflated performance scores, unreliable evaluation",
        "solution": "Machine unlearning to remove contaminated benchmark knowledge",
        "results": "Restored fair evaluation capabilities while preserving general Verilog skills"
    },
    
    "IP Leakage Prevention": {
        "problem": "Model generating proprietary hardware IP designs", 
        "impact": "Trade secret violations, competitive disadvantage",
        "solution": "Selective unlearning of sensitive IP patterns",
        "results": "Model refuses to generate proprietary content while maintaining design skills"
    },
    
    "Malicious Code Mitigation": {
        "problem": "LLM generating hardware trojans or incorrect arithmetic",
        "impact": "Security vulnerabilities, functional failures",
        "solution": "Unlearning malicious patterns, reinforcing correct implementations", 
        "results": "Reduced generation of harmful code while improving correctness"
    }
}

def display_case_study(case_name, case_data):
    """Display a formatted case study"""
    
    print(f"📋 CASE STUDY: {case_name}")
    print("="*60)
    print(f"🔍 Problem: {case_data['problem']}")
    print(f"💥 Impact: {case_data['impact']}")
    print(f"🛡️ Solution: {case_data['solution']}")
    print(f"✅ Results: {case_data['results']}")
    print()

print("🏭 REAL-WORLD APPLICATIONS OF SALAD")
print("="*60)

for case_name, case_data in case_studies.items():
    display_case_study(case_name, case_data)

print("💼 Industry Applications:")
print("• Semiconductor companies protecting IP designs")
print("• EDA tool vendors ensuring benchmark integrity") 
print("• Hardware startups preventing design leakage")
print("• Academic institutions maintaining evaluation fairness")
print("• Security researchers mitigating hardware vulnerabilities")

# 11. Next Steps and Advanced Topics

Congratulations on completing the SALAD tutorial! Here's how you can continue your journey in LLM4ChipDesign.

In [ ]:
# Learning Path and Resources

learning_path = {
    "Beginner": [
        "✅ Complete this SALAD tutorial",
        "📖 Read the SALAD paper (https://arxiv.org/abs/2506.02089)",
        "🎯 Practice Verilog generation with different LLMs",
        "📊 Experiment with basic unlearning techniques"
    ],
    
    "Intermediate": [
        "🔧 Implement your own unlearning algorithms",
        "📈 Design custom evaluation metrics",
        "🏗️ Build end-to-end hardware design pipelines",
        "🔬 Conduct ablation studies on different methods"
    ],
    
    "Advanced": [
        "🧠 Research novel unlearning algorithms for hardware",
        "🏭 Apply to real industrial datasets",
        "📊 Publish benchmarks and evaluation frameworks",
        "🌍 Contribute to open-source LLM4Hardware projects"
    ]
}

additional_resources = {
    "Papers to Read": [
        "SALAD: Systematic Assessment of Machine Unlearning (MLCAD 2024)",
        "RTL-Coder: Outperforming GPT-3.5 in Verilog Generation",
        "Machine Unlearning: A Survey (ACM Computing Surveys)",
        "Large Language Models for Hardware Description"
    ],
    
    "Datasets & Benchmarks": [
        "VerilogEval - Verilog code generation benchmark",
        "RTL-Coder dataset - Hardware description tasks", 
        "TOFU - Task-specific unlearning evaluation",
        "Hardware Trojan benchmarks"
    ],
    
    "Tools & Frameworks": [
        "Hugging Face Transformers - Model implementation",
        "PEFT - Parameter-efficient fine-tuning",
        "Accelerate - Distributed training",
        "EDA tools integration (Vivado, Quartus)"
    ]
}

def print_learning_path():
    """Display the learning path"""
    
    print("🎓 LEARNING PATH FOR LLM4CHIPDESIGN")
    print("="*50)
    
    for level, tasks in learning_path.items():
        print(f"\n🏆 {level.upper()} LEVEL:")
        for task in tasks:
            print(f"   {task}")
    
    print("\n\n📚 ADDITIONAL RESOURCES")
    print("="*50)
    
    for category, items in additional_resources.items():
        print(f"\n📖 {category}:")
        for item in items:
            print(f"   • {item}")

print_learning_path()

print("\n\n🎯 FINAL PROJECT IDEAS:")
print("="*30)
project_ideas = [
    "🔧 Build a complete LLM-based hardware design assistant",
    "🛡️ Develop IP protection system for hardware companies", 
    "📊 Create comprehensive evaluation suite for HW-LLMs",
    "🏭 Apply unlearning to real semiconductor company data",
    "🔍 Research novel unlearning algorithms for hardware domain",
    "🎮 Build interactive hardware design learning platform"
]

for idea in project_ideas:
    print(f"   {idea}")

print("\n\n✨ Thank you for completing the SALAD tutorial!")
print("🚀 You're now ready to explore the frontier of LLM4ChipDesign!")
print("🤝 Connect with the community and keep learning!")